# GenCare AI: Data generatie
---
Auteur:   Eva Rombouts  
Datum:    december 2023  
Script:   Dit script genereert fictieve zorgdata met behulp van de OpenAI API. 

Let op. Het instellen van een OpenAI API key is vereist (https://platform.openai.com/docs/quickstart?context=python)
Het genereren van 24 clientenprofielen met gpt-3.5-turbo kost ongeveer 3 cent. Met gpt-4 krijg je aanzienlijk betere modellen, dan betaal je 70 cent. 
Voor 10 weken rapportages van 24 clienten betaal je 70 cent met 3.5 turbo.

---

11.84
11:15 = 69 cent voor de profielen met gpt-4
10:56 = 19 cent voor de rapportages met gpt-3.5

## Setup

In [1]:
import pandas as pd
import re
import json 
from openai import OpenAI 
import time

In [41]:
seed = 42
aantal_clienten = 24
num_weeks = 10
model_profielen = 'gpt-4'
model_rapportages = 'gpt-3.5-turbo'
afdelingsnaam = 'notenboom'
filename_profielen = f'../zorgdata/{afdelingsnaam}_clienten.csv'
filename_rapportages = f'../zorgdata/{afdelingsnaam}_rapportages.csv'

In [42]:
filename_profielen

'../zorgdata/notenboom_clienten.csv'

## Data generatie functie

Onderstaande functie produceert AI-gegenereerde data.

Allereerst wordt een instantie gemaakt van de OpenAI-client. Hiermee wordt verbinding gemaakt met de OpenAI API.  

Vervolgens genereert chat.completions.create() de tekst.  
Voorbeeld van OpenAI:  
openai.chat.completions.create(  
model="gpt-3.5-turbo",  
  messages=[  
        {"role": "system", "content": "You are a helpful assistant."},  
        {"role": "user", "content": "Who won the world series in 2020?"},  
        {"role": "assistant", "content": "The Los Angeles Dodgers won the World Series in 2020."},  
        {"role": "user", "content": "Where was it played?"}  
    ]  
)

Argumenten:  
- s_role: De system rol bepaalt hoe de 'assistent' zich gedraagt. Je kan instructies meegeven over zijn doel en manier van antwoorden. De system rol wordt slechts één keer gedefinieerd.  
- u_role: De prompt van de 'user' waar de 'assistent' een antwoord (of een completion) op geeft. Eventueel kan dit de start zijn van een dialoog, maar dat gebruiken wij niet. 
- model: Voor mogelijke modellen zie: https://platform.openai.com/docs/models
- seed: spreekt voor zich
- n: Aantal completions die hij teruggeeft

In [3]:
def genereer_zorgdata(s_role, u_role, model='gpt-3.5-turbo', seed=None, n=1):
    try:
        client = OpenAI()
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": s_role},
                {"role": "user", "content": u_role}
            ],
            seed=seed,
            n=n
        )
        return completion
    except Exception as e:
        print(f"Er is een fout opgetreden: {e}")
        return None

## Genereer cliëntprofielen
Definieer inhoud rollen

In [4]:
s_role_profielen = '''
Je specialiseert je in het genereren van fictieve gegevens voor helpend en verzorgend personeel in verpleeghuizen, met een focus op cliënten met een psychogeriatrische aandoening. Jouw expertise omvat het creëren van realistische cliëntscenario's, cliëntendossiers en zorgplannen die specifiek zijn afgestemd op de behoeften van deze doelgroep. Deze gegevens moeten toegankelijk en relevant zijn voor personeel zonder uitgebreide medische kennis, en moeten belangrijke aspecten van de dagelijkse zorg en ondersteuning voor deze cliënten bevatten.
'''

u_role_profielen = '''
Schrijf een profiel van een cliënt die is opgenomen op een psychogeriatrische afdeling van het verpleeghuis. Gebruik onderstaande structuur:

Naam: [Meneer/Mevrouw Voornaam Achternaam]
Type dementie: [kies uit: Alzheimer, gemengde dementie, vasculaire dementie, lewy body dementie, parkinsondementie, FTD, hou rekening met hoe vaak deze vormen van dementie in de populatie voorkomen]
Lichamelijke klachten: [lichamelijke klachten]
Beschrijving cliënt: [een korte beschrijving van karakter en relevante biografische gegevens]
Belangrijkste zorgbehoefte:
- ADL: [Beschrijf ADL hulp]
- cognitie / probleemgedrag: [beschrijf voor de zorg relevante aspecten van cognitie en probleemgedrag. Varieer met de ernst van het probleemgedrag van rustige cliënten, gemiddeld onrustige clienten tot cliënten die fors apathisch, onrustig, angstig, geagiteerd of zelfs agressief kunnen zijn]

Vermijd veelvoorkomende namen als Jansen, de Vries en Fatima.
Vermijd stereotypen in beroepen en achtergronden. Niet elke vrouw was lerares en niet elke man was bankdirecteur of architect.
'''

In [5]:
profielen = genereer_zorgdata(s_role=s_role_profielen, u_role=u_role_profielen, model=model_profielen, seed=seed, n=aantal_clienten)

In [31]:
df_profielen = pd.DataFrame([{"profiel": choice.message.content} for choice in profielen.choices])

In [37]:
df_profielen = (df_profielen
    .assign(client_id=lambda x: x.index.map(lambda i: f'kamer{i+1:02d}'))
    .assign(naam=lambda x: x['profiel'].str.replace('Naam: ', '').str.split('\n').str[0])
    .assign(geslacht=lambda x: x['naam'].apply(lambda y: 'man' if 'Meneer' in y else ('vrouw' if 'Mevrouw' in y else 'onbekend')))
    .assign(naam=lambda x: x['naam'].str.replace('Meneer', '').str.replace('Mevrouw', '').str.strip())
    [['client_id', 'naam', 'geslacht', 'profiel']]
)

In [43]:
df_profielen.to_csv(filename_profielen, index=False)

## Genereer rapportages

In [45]:
# Definieer inhoud rollen
s_role_rapportages = '''
Je specialiseert je nu in het genereren van fictieve gegevens voor helpend en verzorgend personeel in verpleeghuizen, met een focus op cliënten met een psychogeriatrische aandoening. Jouw expertise omvat het creëren van realistische cliëntscenario's, cliëntendossiers en zorgplannen die specifiek zijn afgestemd op de behoeften van deze doelgroep. Deze gegevens moeten toegankelijk en relevant zijn voor personeel zonder uitgebreide medische kennis, en moeten belangrijke aspecten van de dagelijkse zorg en ondersteuning voor deze cliënten bevatten.
'''

u_role_rapportages_instruction = '''
\n
Genereer fictieve zorgrapportages voor deze cliënt voor een periode van zeven dagen. 
De rapportages moeten afwisselend geschreven worden door een helpende (20% kans), een verzorgende (60% kans), of een verpleegkundige (20% kans). Zorg voor variatie in de tijdstippen van de dag waarop de rapportages zijn geschreven, met een focus op overdag, en in mindere mate 's avonds en 's nachts. 
In de inhoud van elke rapportage komt meestal één, soms meer eigenschappen of beperkingen van de cliënt naar voren. Dit kunnen functionele of lichamelijke beperkingen zijn. En soms onrust. Wissel hiermee af, niet elke rapportage gaat over gedrag.

Geef elke rapportage een onrustscore: Een heel getal tussen 0 en 100, wat de mate weergeeft waarin de cliënt onrust vertoont in deze rapportage:
0-20: Geen onrust
21-40: Lichte onrust
41-60: Matige onrust
61-80: Ernstige onrust
81-100: Extreme onrust
Als een rapportage bijvoorbeeld over e

Elke rapportage dient de volgende structuur te hebben: 
---StartRapportage---
Dag: [Dag van de week]
Niveau: [Helpende/Verzorgende/Verpleegkundige]
Onrustscore: [Onrustscore]
Rapportage: [Inhoud van de rapportage]
---EindRapportage---
'''

In [47]:
weekrapportage_list = []
for index, row in df_profielen.iterrows():
    start = time.time()
    client_id = row['client_id']
    print(client_id) # Print om voortgang bij te houden
    # Voeg de instructietekst toe aan het profiel
    u_role_rapportages = row['profiel'] + u_role_rapportages_instruction

    # Genereer de rapportages
    weekrapportages = genereer_zorgdata(s_role=s_role_rapportages, u_role=u_role_rapportages, model=model_rapportages, seed=seed, n=num_weeks)

    # Voeg elke rapportage toe aan de lijst
    for weekrapportage in weekrapportages.choices:
        weekrapportage_list.append({
            'client_id': client_id, 
            'weekno': weekrapportage.index
            'rapportage': weekrapportage.message.content,
        })
    
    print (f"Verstreken tijd: {round(time.time()-start)} seconden")

kamer01
Verstreken tijd: 56 seconden
kamer02
Verstreken tijd: 45 seconden
kamer03
Verstreken tijd: 44 seconden
kamer04
Verstreken tijd: 47 seconden
kamer05
Verstreken tijd: 52 seconden
kamer06
Verstreken tijd: 52 seconden
kamer07
Verstreken tijd: 43 seconden
kamer08
Verstreken tijd: 44 seconden
kamer09
Verstreken tijd: 44 seconden
kamer10
Verstreken tijd: 52 seconden
kamer11
Verstreken tijd: 45 seconden
kamer12
Verstreken tijd: 56 seconden
kamer13
Verstreken tijd: 72 seconden
kamer14
Verstreken tijd: 55 seconden
kamer15
Verstreken tijd: 65 seconden
kamer16
Verstreken tijd: 62 seconden
kamer17
Verstreken tijd: 57 seconden
kamer18
Verstreken tijd: 56 seconden
kamer19
Verstreken tijd: 59 seconden
kamer20
Verstreken tijd: 63 seconden
kamer21
Verstreken tijd: 48 seconden
kamer22
Verstreken tijd: 44 seconden
kamer23
Verstreken tijd: 52 seconden
kamer24
Verstreken tijd: 51 seconden


In [48]:
df_weekrapportages = pd.DataFrame(weekrapportage_list)

In [51]:
df_weekrapportages.to_csv(filename_rapportages, index=False)

In [52]:
def split_rapportage(rapportage_tekst):
    # Combineer start- en eindpatronen in één patroon
    patroon = r'\s*-+\s*(startrapportage|eindrapportage)\s*-+\s*'
    splitpatroon = r'[\n\s]*§+[\n\s]*'

    # Vervang zowel start- als eindpatronen door §
    rapportage = re.sub(patroon, '§', rapportage_tekst, flags=re.IGNORECASE)

    # Verwijder § aan het begin en het einde (indien aanwezig)
    rapportage = re.sub(r'^' + splitpatroon, '', rapportage)
    rapportage = re.sub(splitpatroon + r'$', '', rapportage)

    # Voer een split uit
    rapportages = re.split(splitpatroon, rapportage)
    return rapportages


In [53]:
def parse_dagrapportage(dagrapportage):
    # Zorg dat de onrustscore altijd op een nieuwe regel begint
    rapportage = re.sub(r'\n*(onrustscore)', '\nOnrustscore', dagrapportage, flags=re.IGNORECASE)
    # Zorg dat de rapportage direct achter de tekst 'Rapportage:' begint (dus zonder newline)
    rapportage = re.sub(r'\n*(rapportage:)[\n\s]*', '\nRapportage: ', rapportage, flags=re.IGNORECASE)

    rapportage_delen = re.split(r'\n', rapportage)

    parsed_data = {
        "dag": None,
        "niveau": None,
        "rapportage": None,
        "onrustscore": None
    }

    for deel in rapportage_delen:
        if deel.startswith('Dag:'):
            parsed_data["dag"] = deel[len('Dag:'):].strip().lower()
        elif deel.startswith('Niveau:'):
            parsed_data["niveau"] = deel[len('Niveau:'):].strip()
        # elif deel.startswith('Onrustscore:'):
        #     parsed_data["onrustscore"] = deel[len('Onrustscore:'):].strip()
        elif deel.startswith('Rapportage:'):
            parsed_data["rapportage"] = deel[len('Rapportage:'):].strip()
        elif deel.startswith('Onrustscore:'):
            # Zet de onrustscore om naar een integer
            score = deel[len('Onrustscore:'):].strip()
            try:
                parsed_data["onrustscore"] = int(score)
            except ValueError:
                # Handel eventuele conversiefouten af
                parsed_data["onrustscore"] = None

    return parsed_data

In [76]:
client_rapportages = []

for weekrapportage in weekrapportage_list:
    dagrapportages = split_rapportage(weekrapportage['rapportage'])
    for dagrapportage in dagrapportages:
        parsed_dagrapportage = parse_dagrapportage(dagrapportage)
        parsed_dagrapportage['client_id'] = weekrapportage['client_id']
        client_rapportages.append(parsed_dagrapportage)

In [80]:
df_rapportages = pd.DataFrame(client_rapportages)

In [81]:
df_rapportages.head()

,dag,niveau,rapportage,onrustscore,client_id
0,maandag,Verpleegkundige,Vanochtend heeft meneer Brouwer wat lichte onr...,35,kamer01
1,dinsdag,Helpende,Vandaag heeft meneer Brouwer geen tekenen van ...,15,kamer01
2,woensdag,Verzorgende,Deze ochtend was meneer Brouwer onrustig en ve...,50,kamer01
3,donderdag,Helpende,Vandaag was een rustige dag voor meneer Brouwe...,20,kamer01
4,vrijdag,Verzorgende,Vanochtend was meneer Brouwer erg onrustig en ...,65,kamer01


In [84]:
df_rapportages.to_csv(filename_rapportages, index=False)